# 🧹 Modul 02: Data Cleaning & Preprocessing (Jantung Penambangan Data)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/antonprafanto/data-mining-zero-to-hero/blob/main/notebooks/02_Data_Cleaning_and_Preprocessing.ipynb)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/antonprafanto/data-mining-zero-to-hero/blob/main/notebooks/02_Data_Cleaning_and_Preprocessing.ipynb)

> **Repositori Resmi**: [github.com/antonprafanto/data-mining-zero-to-hero](https://github.com/antonprafanto/data-mining-zero-to-hero)  
> **Tingkat Kesulitan**: Pemula Menengah (From Zero to Hero)  
> **Estimasi Waktu Belajar**: 60 – 75 Menit (Santai dan interaktif)  
> **Tujuan Modul**: Memahami prinsip *Garbage In Garbage Out*, menangani data duplikat, mendeteksi & memvisualisasikan data kosong dengan pustaka `missingno`, memahami teori MCAR/MAR/MNAR, imputasi canggih dengan `KNNImputer`, deteksi pencilan dengan metode IQR & Z-Score, visualisasi efek penskalaan (*MinMaxScaler, StandardScaler, RobustScaler*), serta pengkodean kategori (*OrdinalEncoder & OneHotEncoder*).

---

## 💡 1. Filosofi: Garbage In, Garbage Out

> *"80% waktu seorang praktisi data dihabiskan untuk membersihkan dan menyiapkan data, sedangkan melatih modelnya sendiri hanya butuh 20% waktu."*

Ada pepatah legendaris dalam dunia ilmu komputer:
### 🗑️ **"Garbage In, Garbage Out" (Sampah Masuk, Sampah Pula yang Keluar)**
Jika Anda menyuapi algoritma tercanggih di dunia dengan data yang kotor, bolong-bolong, dobel entri, dan penuh angka aneh, maka hasil prediksi atau rekomendasinya pasti akan **menyesatkan dan membahayakan keputusan bisnis**.

Mari kita bedah 5 pilar utama pembersihan data secara bertahap dan mudah dipahami!

---

## 👥 2. Langkah Pertama: Mendeteksi & Menghapus Data Duplikat

Data ganda/kembar (*duplicate rows*) sering terjadi di dunia nyata akibat pengguna menekan tombol formulir dua kali atau penggabungan dua database yang tumpang tindih.

Mari kita buat dataset nasabah simulasi yang mengandung data kotor, bolong, dan dobel:

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Simulasi data nasabah pengajuan pinjaman bank
data_mentah = {
    'ID_Nasabah': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 2],                      # Ada ID 2 ganda/duplikat!
    'Usia': [25, 34, 28, 45, 120, 32, 29, np.nan, 50, 36, 34],             # Ada NaN dan Outlier 120 tahun
    'Gaji_Juta': [5.0, 7.5, np.nan, 15.0, 12.0, 8.0, 6.5, 4.5, 95.0, 11.0, 7.5],  # Ada NaN dan Outlier 95 Juta
    'Tingkat_Pendidikan': ['S1', 'D3', 'SMA', 'S2', 'S1', 'S1', 'D3', 'SMA', 'S2', np.nan, 'D3'],
    'Status_Pernikahan': ['Lajang', 'Menikah', 'Lajang', np.nan, 'Menikah', 'Lajang', 'Menikah', 'Lajang', 'Menikah', 'Menikah', 'Menikah'],
    'Skor_Kredit': [650, 710, 590, 780, 680, 620, np.nan, 580, 800, 720, 710]
}

df = pd.DataFrame(data_mentah)
print(f"Ukuran Awal Dataset: {df.shape[0]} baris × {df.shape[1]} kolom")

# 1. Memeriksa jumlah baris duplikat
jumlah_duplikat = df.duplicated().sum()
print(f"Jumlah Baris Duplikat: {jumlah_duplikat} baris")

# 2. Menghapus baris duplikat dengan .drop_duplicates()
df = df.drop_duplicates().reset_index(drop=True)
print(f"Ukuran Setelah Duplikat Dihapus: {df.shape[0]} baris")
df

---

## 🩹 3. Penanganan Nilai Hilang (*Missing Values*)

### 📖 3 Jenis Missing Values Menurut Literatur Akademik:
1. **MCAR (*Missing Completely at Random*)**: Data hilang murni karena kebetulan acak (misal: lembar kertas kuesioner basah tertumpah kopi secara acak).
2. **MAR (*Missing at Random*)**: Data hilang memiliki keterkaitan dengan kolom lain yang tercatat (misal: pria cenderung tidak mengisi kolom berat badan).
3. **MNAR (*Missing Not at Random*)**: Data hilang berkaitan erat dengan nilai yang hilang itu sendiri (misal: orang berpenghasilan fantastis sengaja mengosongkan kolom penghasilan karena privasi).

### 🔍 Visualisasi Nilai Kosong dengan Pustaka Standar `missingno`

In [ ]:
import missingno as msno

# Visualisasi matriks data hilang: Garis putih menandakan sel yang bolong/NaN
plt.figure(figsize=(8, 4))
msno.matrix(df, sparkline=False, color=(0.1, 0.4, 0.8))
plt.title('Peta Sebaran Nilai Kosong (Garis Putih = Data Hilang/NaN)', fontsize=13, fontweight='bold')
plt.show()

### 🛠️ Pilihan Teknik Imputasi:
1. **Imputasi Statistik Sederhana (`SimpleImputer`)**:
   - **Median**: Nilai tengah untuk numerik (sangat tahan outlier).
   - **Modus (*most_frequent*)**: Kategori terbanyak untuk data teks.
2. **Imputasi Canggih Berbasis Model (`KNNImputer`)**:
   - Mengisi nilai kosong berdasarkan rata-rata $K$ tetangga yang memiliki profil paling mirip!

In [ ]:
from sklearn.impute import SimpleImputer, KNNImputer

# 1. Mengisi Data Kategorikal dengan MODUS (most_frequent)
imputer_modus = SimpleImputer(strategy='most_frequent')
df[['Tingkat_Pendidikan', 'Status_Pernikahan']] = imputer_modus.fit_transform(df[['Tingkat_Pendidikan', 'Status_Pernikahan']])

# 2. Mengisi Data Numerik Menggunakan KNNImputer (K=3 Tetangga Terdekat)
kolom_numerik = ['Usia', 'Gaji_Juta', 'Skor_Kredit']
knn_imp = KNNImputer(n_neighbors=3)
df[kolom_numerik] = knn_imp.fit_transform(df[kolom_numerik])

print("Status Nilai Kosong Setelah Imputasi (Semua Kolom Bersih!):")
print(df.isna().sum())

---

## 🕵️ 4. Deteksi & Penanganan Pencilan (*Outliers*)

### Mengapa Outlier Berbahaya?
Di data kita ada:
- Nasabah berusia **120 tahun** (salah ketik saat pendaftaran).
- Nasabah bergaji **Rp 95 Juta** (jauh melampaui rata-rata nasabah lain di rentang 5–15 juta).

### A. Deteksi Visual Menggunakan Boxplot

In [ ]:
plt.figure(figsize=(10, 3))
plt.subplot(1, 2, 1)
sns.boxplot(x=df['Usia'], color='lightcoral')
plt.title('Deteksi Outlier Usia (Titik di 120 tahun!)', fontweight='bold')

plt.subplot(1, 2, 2)
sns.boxplot(x=df['Gaji_Juta'], color='skyblue')
plt.title('Deteksi Outlier Gaji (Titik di 95 Juta!)', fontweight='bold')
plt.tight_layout()
plt.show()

### B. Dua Metode Matematis Deteksi Outlier:

#### 1. Metode IQR (*Interquartile Range*) - Terbaik untuk Data Skewed/Miring
- $IQR = Q3 - Q1$
- Batas Bawah = $Q1 - (1.5 \times IQR)$
- Batas Atas = $Q3 + (1.5 \times IQR)$

#### 2. Metode Z-Score - Terbaik untuk Data Distribusi Normal (Simetris)
- $Z = \frac{X - \mu}{\sigma}$
- Data dengan $|Z| > 3$ (atau $|Z| > 2$ pada sampel kecil) resmi dianggap outlier!

In [ ]:
from scipy import stats

# 1. Deteksi dengan Z-Score untuk Kolom Gaji
z_scores_gaji = np.abs(stats.zscore(df['Gaji_Juta']))
outlier_zscore = df[z_scores_gaji > 2.0]  # Ambang batas Z > 2
print("Outlier Terdeteksi via Z-Score pada Kolom Gaji:")
print(outlier_zscore[['ID_Nasabah', 'Usia', 'Gaji_Juta']])

# 2. Deteksi & Capping (Winsorizing) dengan IQR untuk Kolom Usia
Q1_u = df['Usia'].quantile(0.25)
Q3_u = df['Usia'].quantile(0.75)
IQR_u = Q3_u - Q1_u
batas_atas_u = Q3_u + (1.5 * IQR_u)
batas_bawah_u = Q1_u - (1.5 * IQR_u)

# Capping nilai menggunakan .clip() agar tidak menghapus baris nasabah
df['Usia_Bersih'] = df['Usia'].clip(lower=batas_bawah_u, upper=batas_atas_u)
df['Gaji_Bersih'] = df['Gaji_Juta'].clip(lower=df['Gaji_Juta'].quantile(0.05), upper=df['Gaji_Juta'].quantile(0.90))

print(f"\nBatas Maksimal Wajar Usia via IQR: {batas_atas_u:.1f} tahun")
print("Data Usia dan Gaji Setelah Outlier Dibatasi (Capped):")
df[['ID_Nasabah', 'Usia', 'Usia_Bersih', 'Gaji_Juta', 'Gaji_Bersih']].head(5)

---

## ⚖️ 5. Penskalaan Fitur Numerik (*Feature Scaling*)

### Mengapa Scaling Itu Wajib?
Algoritma machine learning berbasis jarak (seperti **K-NN, K-Means Clustering, SVM**) menghitung selisih koordinat Euclidean:  
Jika kolom `Gaji` nilainya jutaan rupiah dan kolom `Usia` hanya puluhan tahun, algoritma akan mengira kolom Gaji bernilai **jutaan kali lebih penting** hanya karena nominal angkanya lebih besar!

### 3 Metode Penskalaan Terpopuler:
| Scaler | Rumus | Rentang Hasil | Karakter Utama |
| :--- | :--- | :--- | :--- |
| **MinMaxScaler** | $\frac{X - X_{min}}{X_{max} - X_{min}}$ | Tepat **0.0 sampai 1.0** | Cocok untuk data yang butuh batas tegas atau jaringan saraf tiruan (*Neural Network*). |
| **StandardScaler** | $\frac{X - \mu}{\sigma}$ | Rata-rata 0, Std 1 | **Pilihan Utama Standar Industri!** Mengubah data menjadi kurva lonceng normal. |
| **RobustScaler** | $\frac{X - Median}{IQR}$ | Berbasis kuartil | **Tahan banting terhadap sisa pencilan/outlier!** |

In [ ]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler

fitur_angka = df[['Usia_Bersih', 'Gaji_Bersih', 'Skor_Kredit']]

# 1. MinMaxScaler (0 s/d 1)
mms = MinMaxScaler()
data_mms = mms.fit_transform(fitur_angka)

# 2. StandardScaler (Mean 0, Std 1)
std = StandardScaler()
data_std = std.fit_transform(fitur_angka)

# 3. RobustScaler (Median & IQR)
rob = RobustScaler()
data_rob = rob.fit_transform(fitur_angka)

print("=== PERBANDINGAN HASIL PENSKALAAN (3 BARIS PERTAMA) ===")
print("1. MinMaxScaler (0 s/d 1):\n", data_mms[:3].round(2))
print("\n2. StandardScaler (Z-Score):\n", data_std[:3].round(2))
print("\n3. RobustScaler:\n", data_rob[:3].round(2))

### 📊 Visualisasi Perbandingan Efek Scaling: Sebelum vs Sesudah

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

# Sebelum Scaling: Skala Gaji mencapai 15 Juta, Usia hanya 50
ax1.scatter(df['Usia_Bersih'], df['Gaji_Bersih'], color='crimson', s=70)
ax1.set_title('SEBELUM Scaling (Skala Tidak Adil)', fontweight='bold')
ax1.set_xlabel('Usia (Puluhan)')
ax1.set_ylabel('Gaji (Jutaan Rupiah)')

# Sesudah StandardScaler: Kedua sumbu berada di skala seragam (-2 s/d +2)
ax2.scatter(data_std[:, 0], data_std[:, 1], color='navy', s=70)
ax2.set_title('SESUDAH StandardScaler (Skala Seragam & Adil)', fontweight='bold')
ax2.set_xlabel('Usia Terstandarisasi')
ax2.set_ylabel('Gaji Terstandarisasi')
ax2.axhline(0, color='grey', linestyle='--', alpha=0.5)
ax2.axvline(0, color='grey', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

---

## 🏷️ 6. Pengkodean Kategori (*Categorical Encoding*)

Algoritma komputer hanya bisa menghitung angka numerik, bukan teks.

Gunakan panduan baku ini:
1. **Ordinal Encoding**: Gunakan jika kategori **memiliki tingkatan/peringkat yang jelas**  
   *Contoh:* Pendidikan: `SMA = 0`, `D3 = 1`, `S1 = 2`, `S2 = 3` menggunakan `OrdinalEncoder`.
2. **One-Hot Encoding**: Gunakan jika kategori **tidak memiliki tingkatan (nominal / setara)**  
   *Contoh:* Status Pernikahan: dibuat kolom biner baru (1 jika Menikah, 0 jika Lajang).  
   *Kiat Emas:* Gunakan parameter `drop_first=True` untuk menghindari jebakan *Dummy Variable Trap* (redundansi korelasi sempurna antar kolom).

In [ ]:
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder

# 1. Ordinal Encoding untuk Tingkat Pendidikan (Hierarki Jelas)
urutan_pendidikan = [['SMA', 'D3', 'S1', 'S2']]
oe = OrdinalEncoder(categories=urutan_pendidikan)
df['Pendidikan_Encoded'] = oe.fit_transform(df[['Tingkat_Pendidikan']])

# 2. One-Hot Encoding dengan Scikit-Learn OneHotEncoder (Atau pd.get_dummies)
ohe = OneHotEncoder(drop='first', sparse_output=False)
status_ohe = ohe.fit_transform(df[['Status_Pernikahan']])
df['Status_Menikah'] = status_ohe

print("=== DATASET AKHIR YANG 100% BERSIH, TERSTANDARISASI & SIAP DILATIH! ===")
kolom_final = ['ID_Nasabah', 'Usia_Bersih', 'Gaji_Bersih', 'Skor_Kredit', 'Pendidikan_Encoded', 'Status_Menikah']
df[kolom_final]

---

## 🧠 7. Kuis Evaluasi Pemahaman Mandiri

Uji pemahaman Anda terhadap seluruh pilar pembersihan data di Modul 2:

In [ ]:
# Evaluasi Pemahaman Mandiri Modul 2
evaluasi_m2 = [
    {
        'no': 1,
        'soal': 'Mengapa mengisi nilai kosong numerik dengan MEDIAN lebih disarankan daripada MEAN jika data memiliki outlier?',
        'jawaban': 'Karena Median adalah nilai tengah yang kebal terhadap angka ekstrem, sedangkan Mean akan tertarik drastis oleh angka outlier.'
    },
    {
        'no': 2,
        'soal': 'Bagaimana cara kerja algoritma KNNImputer dalam mengisi data kosong?',
        'jawaban': 'KNNImputer mencari K baris data tetangga terdekat yang profilnya paling mirip, lalu mengambil rata-rata nilai fitur tersebut.'
    },
    {
        'no': 3,
        'soal': 'Apa rumus batas atas outlier menggunakan metode IQR?',
        'jawaban': 'Batas Atas = Q3 + (1.5 × IQR). Data di atas angka ini dianggap outlier.'
    },
    {
        'no': 4,
        'soal': 'Mengapa algoritma berbasis jarak seperti K-NN dan K-Means mewajibkan penskalaan fitur (Scaling)?',
        'jawaban': 'Agar fitur dengan rentang angka besar (misal gaji jutaan) tidak mendominasi perhitungan jarak dibanding fitur berangka kecil (misal usia puluhan).'
    },
    {
        'no': 5,
        'soal': 'Mengapa kita perlu menambahkan parameter drop_first=True pada One-Hot Encoding?',
        'jawaban': 'Untuk menghindari Dummy Variable Trap (multikolinearitas sempurna antar kolom biner).'
    }
]

for ev in evaluasi_m2:
    print(f"Soal {ev['no']}: {ev['soal']}")
    print(f"👉 Kunci: {ev['jawaban']}\n")
print("Jika Anda memahami ke-5 pilar di atas, Anda telah menguasai jantung penambangan data! 🌟")

---

## ✍️ 8. Kotak Latihan Bebas (*Playground Challenge*)

Selesaikan tantangan terpandu di bawah ini untuk menguji keahlian baru Anda:

In [ ]:
# TANTANGAN MANDIRI:
# Diberikan data tinggi badan pasien rumah sakit berikut:
tinggi_pasien = pd.Series([160, 165, np.nan, 170, 155, 290, 168, 165]) # Ada 1 NaN, 1 Outlier ekstrem (290 cm), dan duplikat!

# 1. Hapus nilai duplikat
tinggi_tanpa_duplikat = tinggi_pasien.drop_duplicates()

# 2. Isi missing value dengan nilai Median
tinggi_imputasi = tinggi_tanpa_duplikat.fillna(tinggi_tanpa_duplikat.median())

# 3. Batasi nilai maksimal wajar di angka 190 cm menggunakan .clip()
tinggi_final = tinggi_imputasi.clip(upper=190)

print("Hasil Pembersihan Tinggi Pasien:")
print(tinggi_final.reset_index(drop=True))
print(f"Rata-rata Tinggi Pasien Bersih: {tinggi_final.mean():.1f} cm ✅")

---

## 🎯 9. Checklist Kelulusan Modul 02 (*Hero Checkpoint*)

Beri tanda centang pada hati Anda jika Anda sudah menguasai poin-poin berikut:
- [ ] Saya paham pepatah *Garbage In, Garbage Out*.
- [ ] Saya bisa mendeteksi dan menghapus baris duplikat dengan `.drop_duplicates()`.
- [ ] Saya tahu perbedaan konsep data hilang: MCAR, MAR, dan MNAR.
- [ ] Saya bisa memvisualisasikan data kosong menggunakan pustaka `missingno`.
- [ ] Saya bisa mengisi missing values dengan Median, Modus, dan `KNNImputer`.
- [ ] Saya bisa mendeteksi outlier menggunakan Boxplot, rumus IQR, dan metode Z-Score.
- [ ] Saya bisa menerapkan teknik pembatasan nilai (*Capping / Winsorizing*) dengan `.clip()`.
- [ ] Saya paham mengapa fitur numerik wajib disetarakan skalanya (*Feature Scaling*).
- [ ] Saya bisa membedakan MinMaxScaler, StandardScaler, dan RobustScaler.
- [ ] Saya bisa membedakan Ordinal Encoding (bertingkat) dan One-Hot Encoding (nominal setara).
- [ ] Saya paham bahaya *Dummy Variable Trap* dan cara mengatasinya dengan `drop_first=True`.

---

## 🏁 Gerbang Menuju Modul 03: EDA & Feature Engineering!

Kerja yang luar biasa! Data Anda kini sudah bersih, bebas dari duplikat dan outlier, rapi, dan memiliki format angka yang adil untuk dilatih oleh komputer.

Langkah berikutnya adalah **mengekstrak wawasan tersembunyi dan menciptakan variabel-variabel baru yang lebih cerdas**:

👉 **[Buka Modul 03: Exploratory Data Analysis & Feature Engineering](03_EDA_and_Feature_Engineering.ipynb)**

Sampai jumpa di Modul 03! Tetap semangat dan jadilah Hero Data Mining! 🚀